# ManimFlow

Autonomous Manim animation generator powered by Pi and Exa. No API key is required.

Run the setup cell once, then use the generator below. Pi plans, writes the project, runs Manim, diagnoses failures, and continues for up to six attempts.


In [ ]:
import os, pathlib, platform, shutil, subprocess, sys, urllib.request

BASE = pathlib.Path.cwd()
NODE_DIR = BASE / 'node22'
PI_DIR = BASE / 'pi-runtime'
RUNNER = BASE / 'manimflow_pi.py'
EXA_EXTENSION = BASE / 'exa_direct.ts'
RAW_BASE = 'https://raw.githubusercontent.com/theabbie/ManimFlow/main'

if platform.system() == 'Linux':
    packages = ['libcairo2-dev', 'libpango1.0-dev', 'ffmpeg', 'ripgrep', 'fd-find',
                'texlive-latex-base', 'texlive-latex-extra', 'texlive-fonts-recommended',
                'texlive-plain-generic', 'dvisvgm']
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-q', *packages], check=True)
    if not pathlib.Path('/usr/local/bin/fd').exists() and shutil.which('fdfind'):
        pathlib.Path('/usr/local/bin/fd').symlink_to(shutil.which('fdfind'))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy==1.26.4', 'scipy==1.13.1', 'manim', 'nodeenv'], check=True)
if not (NODE_DIR / 'bin' / 'node').exists():
    subprocess.run([sys.executable, '-m', 'nodeenv', '--node=22.19.0', '--prebuilt', str(NODE_DIR)], check=True)
npm = NODE_DIR / 'bin' / 'npm'
subprocess.run([str(npm), 'install', '--prefix', str(PI_DIR), '--no-audit', '--no-fund',
                '@earendil-works/pi-coding-agent@0.80.6'], check=True)
urllib.request.urlretrieve(f'{RAW_BASE}/manimflow_pi.py', RUNNER)
urllib.request.urlretrieve(f'{RAW_BASE}/pi_extensions/exa_direct.ts', EXA_EXTENSION)
print('Manim and Pi are ready.')


In [ ]:
import IPython.display

MANIM_OUTPUT = BASE / 'manim_output'
VIDEO = BASE / 'animation.mp4'
PI = PI_DIR / 'node_modules' / '.bin' / 'pi'

def run_manimator(prompt):
    if not prompt.strip():
        raise ValueError('Enter an animation prompt.')
    command = [sys.executable, str(RUNNER), '--provider', 'exa', '--prompt', prompt.strip(),
               '--workspace', str(MANIM_OUTPUT), '--output', str(VIDEO), '--pi', str(PI),
               '--exa-extension', str(EXA_EXTENSION), '--attempts', '6']
    subprocess.run(command, check=True, timeout=3600)
    IPython.display.display(IPython.display.Video(str(VIDEO), embed=True, html_attributes='controls width="720"'))

print('run_manimator() is ready.')


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

prompt_box = widgets.Textarea(placeholder='Describe the animation...', layout=widgets.Layout(width='100%', height='110px'))
generate_button = widgets.Button(description='Generate', button_style='primary')
output_box = widgets.Output()

def generate(_):
    with output_box:
        clear_output(wait=True)
        try:
            run_manimator(prompt_box.value)
        except Exception as error:
            print(f'Generation failed: {error}')

generate_button.on_click(generate)
display(prompt_box, generate_button, output_box)
